# 02 Train — 多轮 SFT 数据 → LoRA

TaleTalk 流水线第二阶段：吃 `01_extract.ipynb` 产出的 ShareGPT 多轮数据，用 LLaMA Factory 在 ROCm AMD GPU 上做 LoRA 微调。

**这个 notebook 假设运行在 Radeon Cloud 之类的 ROCm 环境**，本地 macOS 跑不动。

## 0. 参数区

`RUN_NAME` 必须和 `01_extract.ipynb` 一致，这样能直接读到生成的数据。`RUN_TAG` 留空就 fresh 训，填了会拼到 `output_dir` 末尾用来 A/B（比如 `r16`、`bs8`）。

In [ ]:
from pathlib import Path
import os, sys, json, subprocess

# ====== 数据集 ======
RUN_NAME = 'shiri_qixia'        # 必须和 01_extract 一致
TARGET_ROLE = '齐夏'             # 仅用于 system prompt 文案（数据里已经写好了 system）
NOVEL_TITLE = '十日终焉'

# ====== 模型 ======
MODEL_IDS = {
    'qwen3_5_9b': 'Qwen/Qwen3.5-9B',
    'qwen3_6_35b_a3b': 'Qwen/Qwen3.6-35B-A3B',
}
MODEL_CHOICE = 'qwen3_5_9b'
MODEL_ID = MODEL_IDS[MODEL_CHOICE]

# ====== 训练超参 ======
TRAIN_OVERRIDES = {
    'per_device_train_batch_size': 4,
    'gradient_accumulation_steps': 4,
    'learning_rate': 1.0e-4,
    'num_train_epochs': 2.0,
    'lora_rank': 8,
    'lora_alpha': 16,
    'lora_dropout': 0.05,
    'cutoff_len': 1024,
    'warmup_ratio': 0.05,
    'lr_scheduler_type': 'cosine',
    'logging_steps': 5,
    'save_steps': 100,
    'eval_steps': 100,
    'gradient_checkpointing': False,
}

RUN_TAG = ''               # A/B 实验留个后缀，避免覆盖旧 LoRA
RUN_INSTALL = True         # 是否跑 pip install（云端首次必须 True）
RUN_TRAIN = False          # 检查无误后改 True 再跑这一格之后的训练 cell

# ====== 路径 ======
REPO_DIR = Path.cwd().resolve()
while REPO_DIR != REPO_DIR.parent and not (REPO_DIR / 'extract').is_dir():
    REPO_DIR = REPO_DIR.parent
assert (REPO_DIR / 'extract').is_dir(), 'taletalk 仓库根目录找不到'

PERSIST_DIR = Path('/network-workspace')
DATA_DIR = REPO_DIR / "data"
CONFIG_PATH = REPO_DIR / 'configs' / f'{MODEL_CHOICE}_lora.yaml'
FULL_RUN_NAME = f"{RUN_NAME}_{MODEL_CHOICE}_lora" + (f"_{RUN_TAG}" if RUN_TAG else "")
OUTPUT_DIR = PERSIST_DIR / "outputs" / FULL_RUN_NAME
LOG_DIR = PERSIST_DIR / "logs"
MODEL_CACHE_DIR = PERSIST_DIR / "models"
RUNTIME_CONFIG_DIR = PERSIST_DIR / "runtime_configs"
RUNTIME_CONFIG_PATH = RUNTIME_CONFIG_DIR / f"{FULL_RUN_NAME}.yaml"

TRAIN_JSON = DATA_DIR / f'{RUN_NAME}_chat_train.json'
VALID_JSON = DATA_DIR / f'{RUN_NAME}_chat_valid.json'
DATASET_INFO = DATA_DIR / 'dataset_info.json'

for p in [PERSIST_DIR, OUTPUT_DIR, LOG_DIR, MODEL_CACHE_DIR, RUNTIME_CONFIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

assert TRAIN_JSON.exists(), f'缺训练数据: {TRAIN_JSON}（先在 01_extract.ipynb 里把数据跑出来）'
assert VALID_JSON.exists(), f'缺验证数据: {VALID_JSON}'
assert DATASET_INFO.exists(), f'缺 dataset_info.json: {DATASET_INFO}'
assert CONFIG_PATH.exists(), f'缺配置文件: {CONFIG_PATH}'

print('python:', sys.version.split()[0])
print('repo:', REPO_DIR)
print('persist:', PERSIST_DIR)
print('run_name:', RUN_NAME, '/ full:', FULL_RUN_NAME)
print('model:', MODEL_ID)
print('config:', CONFIG_PATH)
print('runtime config:', RUNTIME_CONFIG_PATH)
print('train data:', TRAIN_JSON)
print('valid data:', VALID_JSON)
print('output:', OUTPUT_DIR)
print('overrides:')
for k, v in TRAIN_OVERRIDES.items():
    print(f'  {k}: {v}')
eff = TRAIN_OVERRIDES['per_device_train_batch_size'] * TRAIN_OVERRIDES['gradient_accumulation_steps']
print(f'  effective_batch_size: {eff}')

## 1. 命令工具

In [ ]:
def run(cmd, cwd=None, timeout=None):
    if cwd is None:
        cwd = REPO_DIR
    print(f'$ {cmd}', flush=True)
    process = subprocess.Popen(
        cmd, shell=True, cwd=cwd, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
    )
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        returncode = process.wait(timeout=timeout)
    except Exception:
        process.kill(); raise
    if returncode != 0:
        raise RuntimeError(f'command failed with exit code {returncode}: {cmd}')
    return returncode

## 2. ROCm / PyTorch 自检

必须看到 `torch.version.hip` 不为空，`torch.cuda.is_available()` 为 True。

In [ ]:
run('rocm-smi --showproductname --showmeminfo vram --showuse || true', timeout=60)

import torch
print('torch:', torch.__version__)
print('hip:', getattr(torch.version, 'hip', None))
print('cuda available:', torch.cuda.is_available())
print('device count:', torch.cuda.device_count())
assert getattr(torch.version, 'hip', None), '当前不是 ROCm/HIP 版 PyTorch'
assert torch.cuda.is_available(), 'PyTorch 没识别到 AMD GPU'

props = torch.cuda.get_device_properties(0)
print('device:', torch.cuda.get_device_name(0))
print('total_memory_gb:', round(props.total_memory / 1024**3, 2))
print('bf16 supported:', torch.cuda.is_bf16_supported())

## 3. 安装训练依赖

`requirements-rocm.txt` 不含 torch，避免覆盖云镜像已有的 ROCm PyTorch。

In [ ]:
REQ_PATH = REPO_DIR / 'requirements-rocm.txt'
assert REQ_PATH.exists(), f'缺 {REQ_PATH}'

if RUN_INSTALL:
    run('python -m pip install -U pip setuptools wheel', timeout=300)
    run(f'python -m pip install -r {REQ_PATH} --upgrade-strategy only-if-needed', timeout=1200)
else:
    print('skip install')

run('python scripts/patch_llamafactory_qwen35_text.py', timeout=60)

import torch
print('torch after install:', torch.__version__)
assert getattr(torch.version, 'hip', None), '安装依赖后 torch 不再是 ROCm 版'

## 4. 数据校验

In [ ]:
run(f'python scripts/validate_dataset.py {TRAIN_JSON} {VALID_JSON}', timeout=120)

train = json.loads(TRAIN_JSON.read_text(encoding="utf-8"))
valid = json.loads(VALID_JSON.read_text(encoding="utf-8"))
print('train samples:', len(train))
print('valid samples:', len(valid))

# 看一个多轮样本
multi = [s for s in train if len(s["conversations"]) >= 4]
print(f'多轮样本数（>=4 turns）: {len(multi)} / {len(train)}')
if multi:
    print(json.dumps(multi[0], ensure_ascii=False, indent=2)[:1500])

## 5. 训练配置检查 + 下载模型

从 ModelScope 下载 base model（HF 在云端通常不通），然后用 `TRAIN_OVERRIDES` 把 yaml 默认值覆盖掉，写成 runtime config。

In [ ]:
def replace_yaml_value(text, key, value):
    if isinstance(value, float):
        formatted = repr(value)
    else:
        formatted = str(value)
    lines, replaced = [], False
    for line in text.splitlines():
        if line.startswith(f'{key}:'):
            lines.append(f'{key}: {formatted}'); replaced = True
        else:
            lines.append(line)
    if not replaced:
        lines.append(f'{key}: {formatted}')
    return "\n".join(lines) + "\n"

print('downloading base model from ModelScope:', MODEL_ID)
from modelscope import snapshot_download
MODEL_DIR = Path(snapshot_download(MODEL_ID, cache_dir=str(MODEL_CACHE_DIR))).resolve()
assert (MODEL_DIR / 'config.json').exists(), f'模型 config.json 缺失: {MODEL_DIR}'
print('model_dir:', MODEL_DIR)

runtime_config = CONFIG_PATH.read_text(encoding='utf-8')
runtime_config = replace_yaml_value(runtime_config, 'model_name_or_path', MODEL_DIR)
runtime_config = replace_yaml_value(runtime_config, 'dataset', f'{RUN_NAME}_train')
runtime_config = replace_yaml_value(runtime_config, 'eval_dataset', f'{RUN_NAME}_valid')
runtime_config = replace_yaml_value(runtime_config, 'dataset_dir', DATA_DIR)
runtime_config = replace_yaml_value(runtime_config, 'output_dir', OUTPUT_DIR)
runtime_config = replace_yaml_value(runtime_config, 'logging_dir', LOG_DIR)

for key, value in TRAIN_OVERRIDES.items():
    runtime_config = replace_yaml_value(runtime_config, key, value)

RUNTIME_CONFIG_PATH.write_text(runtime_config, encoding='utf-8')
print('runtime config:')
print(RUNTIME_CONFIG_PATH.read_text(encoding='utf-8'))

## 6. 开始 LoRA 训练

确认前面都通过后，把参数区的 `RUN_TRAIN = True` 再跑这一格。

训练耗时取决于数据量、序列长度、模型大小。9B + cutoff=1024 + 6k 多轮样本，单卡 W7900D 约 1-2 小时。

In [ ]:
if RUN_TRAIN:
    env = os.environ.copy()
    env['HIP_VISIBLE_DEVICES'] = '0'
    env['CUDA_VISIBLE_DEVICES'] = '0'
    env['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
    env['TOKENIZERS_PARALLELISM'] = 'false'
    env['PYTHONUNBUFFERED'] = '1'
    run(f'python scripts/train_lora.py --config {RUNTIME_CONFIG_PATH}', timeout=None)
else:
    print('skip train. Set RUN_TRAIN=True 后再跑这格')

## 7. 打包 LoRA 供下载

把最新 checkpoint 打包到 `downloads/`，左边文件浏览器直接点下载。

In [ ]:
import re, shutil, tarfile

def checkpoint_step(p):
    m = re.search(r'checkpoint-(\d+)$', p.name)
    return int(m.group(1)) if m else -1

checkpoints = [p for p in OUTPUT_DIR.glob('checkpoint-*') if (p / 'adapter_model.safetensors').exists()]
ADAPTER_DIR = max(checkpoints, key=checkpoint_step) if checkpoints else OUTPUT_DIR
assert (ADAPTER_DIR / 'adapter_model.safetensors').exists(), f'缺 adapter: {ADAPTER_DIR}'

DOWNLOAD_DIR = REPO_DIR / 'downloads'
PACKAGE_DIR = DOWNLOAD_DIR / f'{FULL_RUN_NAME}_{ADAPTER_DIR.name}'
TAR_PATH = DOWNLOAD_DIR / f'{FULL_RUN_NAME}_{ADAPTER_DIR.name}.tar.gz'

if PACKAGE_DIR.exists():
    shutil.rmtree(PACKAGE_DIR)
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

files = [
    'adapter_config.json', 'adapter_model.safetensors',
    'chat_template.jinja', 'processor_config.json',
    'tokenizer.json', 'tokenizer_config.json', 'README.md',
]
for name in files:
    src = ADAPTER_DIR / name
    if src.exists():
        shutil.copy2(src, PACKAGE_DIR / name)
with tarfile.open(TAR_PATH, 'w:gz') as tar:
    tar.add(PACKAGE_DIR, arcname=PACKAGE_DIR.name)
print('adapter:', ADAPTER_DIR)
print('package:', PACKAGE_DIR)
print('download:', TAR_PATH)
print('size_mb:', round(TAR_PATH.stat().st_size / 1024 / 1024, 2))

## 8. 完成

打开 `03_infer.ipynb`，把 `RUN_NAME` 和 `MODEL_CHOICE` 改成和这里一致，就能加载训练好的 LoRA 起 Gradio 多轮对话。